# RQ3: Effect of Preprocessing
**Research Question:** How do different data preprocessing strategies affect the performance of supervised learning models on the Marketing and Product Performance Dataset?

**Strategies tested:** Raw data → Imputation → Scaling + Encoding → Full pipeline (+ outlier removal)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

In [2]:
DATA_PATH = '/kaggle/input/datasets/karansridhar/karan-ue-ml-assignment-dataset/marketing_and_product_performance.csv'
df_raw = pd.read_csv(DATA_PATH)
TARGET = 'Revenue_Generated'

# Remove ID cols
id_cols = [c for c in df_raw.columns if 'ID' in c or 'id' in c.lower()]
df_raw = df_raw.drop(columns=id_cols, errors='ignore')
print('Shape after dropping IDs:', df_raw.shape)

Shape after dropping IDs: (10000, 12)


In [3]:
def evaluate(X_train, X_test, y_train, y_test):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return {
        'MAE':  round(mean_absolute_error(y_test, preds), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, preds)), 4),
        'R²':   round(r2_score(y_test, preds), 4)
    }

results = []

# --- Strategy 1: Raw data (drop NaN rows, basic encode) ---
df1 = df_raw.dropna(subset=[TARGET]).copy()
cat_cols = df1.select_dtypes(include='object').columns
le = LabelEncoder()
for c in cat_cols:
    df1[c] = le.fit_transform(df1[c].astype(str))
df1 = df1.dropna()
X1, y1 = df1.drop(columns=[TARGET]), df1[TARGET]
Xtr1, Xte1, ytr1, yte1 = train_test_split(X1, y1, test_size=0.2, random_state=42)
r1 = evaluate(Xtr1, Xte1, ytr1, yte1)
r1['Strategy'] = 'Raw Data'
results.append(r1)
print('Strategy 1 done:', r1)

# --- Strategy 2: Imputation ---
df2 = df_raw.copy()
cat_cols = df2.select_dtypes(include='object').columns
for c in cat_cols:
    df2[c] = le.fit_transform(df2[c].fillna('Unknown').astype(str))
df2 = df2.dropna(subset=[TARGET])
X2, y2 = df2.drop(columns=[TARGET]), df2[TARGET]
imputer = SimpleImputer(strategy='median')
X2_imp = pd.DataFrame(imputer.fit_transform(X2), columns=X2.columns)
Xtr2, Xte2, ytr2, yte2 = train_test_split(X2_imp, y2, test_size=0.2, random_state=42)
r2_ = evaluate(Xtr2, Xte2, ytr2, yte2)
r2_['Strategy'] = 'Imputation'
results.append(r2_)
print('Strategy 2 done:', r2_)

# --- Strategy 3: Scaling + Encoding ---
scaler = StandardScaler()
X3_sc = pd.DataFrame(scaler.fit_transform(X2_imp), columns=X2_imp.columns)
Xtr3, Xte3, ytr3, yte3 = train_test_split(X3_sc, y2, test_size=0.2, random_state=42)
r3 = evaluate(Xtr3, Xte3, ytr3, yte3)
r3['Strategy'] = 'Scaling + Encoding'
results.append(r3)
print('Strategy 3 done:', r3)

# --- Strategy 4: Full pipeline (+ IQR outlier removal) ---
df4 = df2.copy()
df4[TARGET] = df4[TARGET].clip(
    lower=df4[TARGET].quantile(0.01),
    upper=df4[TARGET].quantile(0.99)
)
X4, y4 = df4.drop(columns=[TARGET]), df4[TARGET]
X4_imp = pd.DataFrame(imputer.fit_transform(X4), columns=X4.columns)
X4_sc  = pd.DataFrame(scaler.fit_transform(X4_imp), columns=X4_imp.columns)
Xtr4, Xte4, ytr4, yte4 = train_test_split(X4_sc, y4, test_size=0.2, random_state=42)
r4 = evaluate(Xtr4, Xte4, ytr4, yte4)
r4['Strategy'] = 'Full Pipeline'
results.append(r4)
print('Strategy 4 done:', r4)

Strategy 1 done: {'MAE': 25035.6351, 'RMSE': np.float64(29029.5867), 'R²': -0.0335, 'Strategy': 'Raw Data'}
Strategy 2 done: {'MAE': 25035.6351, 'RMSE': np.float64(29029.5867), 'R²': -0.0335, 'Strategy': 'Imputation'}
Strategy 3 done: {'MAE': 25039.0921, 'RMSE': np.float64(29029.0909), 'R²': -0.0334, 'Strategy': 'Scaling + Encoding'}
Strategy 4 done: {'MAE': 25011.1184, 'RMSE': np.float64(28994.7823), 'R²': -0.0328, 'Strategy': 'Full Pipeline'}


In [4]:
df_results = pd.DataFrame(results)[['Strategy','MAE','RMSE','R²']]
df_results.to_csv('tables/RQ3_Table3_Preprocessing_Effect.csv', index=False)
print(df_results)

             Strategy         MAE        RMSE      R²
0            Raw Data  25035.6351  29029.5867 -0.0335
1          Imputation  25035.6351  29029.5867 -0.0335
2  Scaling + Encoding  25039.0921  29029.0909 -0.0334
3       Full Pipeline  25011.1184  28994.7823 -0.0328


In [5]:
# Figure 3 — Ablation bar chart
x = np.arange(len(df_results))
width = 0.25
colors = ['#5C6BC0', '#EF5350', '#26A69A']

fig, ax = plt.subplots(figsize=(11, 6))
for i, (metric, color) in enumerate(zip(['MAE','RMSE','R²'], colors)):
    ax.bar(x + i*width, df_results[metric], width, label=metric, color=color, edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels(df_results['Strategy'], fontsize=11, rotation=10)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Figure 3: Performance Gains from Preprocessing Strategies\n(Marketing and Product Performance Dataset)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/RQ3_Figure3_Preprocessing_Effect.pdf', bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.
